# 03 — Anomaly Detection
### PM Accelerator — Weather Trend Forecasting Assessment

This notebook:
- Selects key weather features for anomaly detection
- Scales features with StandardScaler
- Fits **IsolationForest** (contamination=0.05)
- Labels anomalies in the DataFrame
- Scatter plot of temperature over time colored by anomaly status
- Prints top 10 anomalous rows for inspection

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from src.utils import DATA_DIR, FIGURES_DIR, apply_plot_style, save_figure

apply_plot_style()
%matplotlib inline

In [2]:
df = pd.read_csv(
    os.path.join(DATA_DIR, 'cleaned_weather_unscaled.csv'),
    parse_dates=['last_updated'],
)
# Reset index to avoid duplicate datetime issues
df.sort_values('last_updated', inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Shape: {df.shape}')

Shape: (130783, 42)


## Feature Selection
Select features for anomaly detection, dropping any with >20% missing data.

In [3]:
anomaly_features = ['temperature_celsius', 'precip_mm', 'wind_mph', 'humidity', 'pressure_mb']

available = []
for col in anomaly_features:
    if col in df.columns:
        pct_missing = df[col].isnull().mean()
        if pct_missing <= 0.20:
            available.append(col)
            print(f'  ✔ {col} ({pct_missing*100:.1f}% missing)')
        else:
            print(f'  ✗ {col} dropped ({pct_missing*100:.1f}% missing)')

print(f'\nUsing {len(available)} features: {available}')

  ✔ temperature_celsius (0.0% missing)
  ✔ precip_mm (0.0% missing)
  ✔ wind_mph (0.0% missing)
  ✔ humidity (0.0% missing)
  ✔ pressure_mb (0.0% missing)

Using 5 features: ['temperature_celsius', 'precip_mm', 'wind_mph', 'humidity', 'pressure_mb']


## Fit Isolation Forest

In [4]:
X = df[available].dropna()
print(f'Rows used for anomaly detection: {len(X)}')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso_forest = IsolationForest(
    contamination=0.05,
    random_state=42,
    n_estimators=100,
    n_jobs=-1,
)
preds = iso_forest.fit_predict(X_scaled)

# Label anomalies — use integer index to avoid duplicate datetime issues
df['anomaly'] = 1  # default: normal
df.loc[X.index, 'anomaly'] = preds

anomaly_count = (df['anomaly'] == -1).sum()
normal_count = (df['anomaly'] == 1).sum()
total = len(df)

print(f'\n{"="*50}')
print(f'ANOMALY DETECTION RESULTS')
print(f'{"="*50}')
print(f'Normal:    {normal_count:>8,} ({normal_count/total*100:.2f}%)')
print(f'Anomaly:   {anomaly_count:>8,} ({anomaly_count/total*100:.2f}%)')
print(f'Total:     {total:>8,}')

Rows used for anomaly detection: 130783

ANOMALY DETECTION RESULTS
Normal:     124,243 (95.00%)
Anomaly:      6,540 (5.00%)
Total:      130,783


## Anomaly Scatter Plot
Temperature over time, with anomalies highlighted in red.

In [5]:
fig, ax = plt.subplots(figsize=(14, 6))

normals = df[df['anomaly'] == 1]
anomalies = df[df['anomaly'] == -1]

if len(normals) > 20000:
    normals_plot = normals.sample(20000, random_state=42)
else:
    normals_plot = normals

ax.scatter(normals_plot['last_updated'], normals_plot['temperature_celsius'],
           c='#3498DB', s=2, alpha=0.3, label='Normal', edgecolors='none')
ax.scatter(anomalies['last_updated'], anomalies['temperature_celsius'],
           c='#E74C3C', s=8, alpha=0.7, label='Anomaly', edgecolors='none')

ax.set_title('Temperature Over Time — Anomalies Detected by Isolation Forest',
             fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
ax.legend(loc='upper right', fontsize=12, markerscale=3)
ax.grid(True, alpha=0.3)
save_figure(fig, '07_anomaly_detection.png')
plt.show()

Figure saved → c:\Users\purva\OneDrive\Desktop\PMA\outputs\figures\07_anomaly_detection.png


## Top 10 Anomalous Rows

In [6]:
anomaly_df = df[df['anomaly'] == -1][available + ['country', 'location_name']].copy()
anomaly_df['temp_abs'] = anomaly_df['temperature_celsius'].abs()
top10 = anomaly_df.nlargest(10, 'temp_abs').drop(columns=['temp_abs'])
display(top10)

,temperature_celsius,precip_mm,wind_mph,humidity,pressure_mb,country,location_name
6918,49.2,0.0,13.6,4,996.0,Kuwait,Kuwait City
7512,49.1,0.0,6.9,8,994.0,Iraq,Baghdad
11180,49.1,0.0,12.8,6,999.0,Iraq,Baghdad
8455,48.9,0.0,13.6,4,993.0,Kuwait,Kuwait City
11370,48.8,0.0,22.4,6,995.0,Iraq,Baghdad
85071,48.8,0.0,2.2,5,995.0,Kuwait,Kuwait City
76883,48.6,0.0,8.1,4,996.0,Kuwait,Kuwait City
7312,48.4,0.0,10.5,8,998.0,Iraq,Baghdad
10004,48.3,0.0,21.7,6,995.0,Iraq,Baghdad
88202,48.2,0.0,21.5,7,1000.0,Kuwait,Kuwait City


In [7]:
# Save updated dataframe
output_path = os.path.join(DATA_DIR, 'weather_with_anomalies.csv')
df.to_csv(output_path, index=False)
print(f'Saved data with anomaly labels → {output_path}')
print('\n✅ Anomaly detection complete!')

Saved data with anomaly labels → c:\Users\purva\OneDrive\Desktop\PMA\data\weather_with_anomalies.csv

✅ Anomaly detection complete!
